# FROST Preprocessing

This notebook explains the **preprocessing stage** used before signing in FROST.

This stage comes after [03-keygen.ipynb](03-keygen.ipynb). Key generation gives each participant a long-lived secret share $s_i$. Preprocessing prepares the **single-use nonce material** that will later be consumed during signing.

The goal is to move work out of the online signing step. Instead of generating nonces only when a message arrives, participants can prepare batches of nonce commitments in advance.

## Why Preprocessing Exists

FROST signing is designed to be efficient during the actual signing moment. Preprocessing helps by letting participants generate and publish commitment material ahead of time.

This has two main benefits:

- less online coordination when a real message needs to be signed
- better support for asynchronous or network-limited participants

The paper presents FROST as a preprocessing phase plus a later single-round signing phase. These can also be combined into a simpler two-round signing protocol if an implementation does not want to cache commitments.

## What Each Participant Generates

For each participant $P_i$, preprocessing generates a batch of nonce pairs and their public commitments.

Let $\pi$ be the number of nonce/commitment pairs generated at one time. Then participant $P_i$ creates a list indexed by $j = 1, \dots, \pi$.

For each $j$, participant $P_i$ samples two single-use nonces:

$$
(d_{ij}, e_{ij}) \xleftarrow{\$} \mathbb{Z}_q^* \times \mathbb{Z}_q^*
$$

and derives the corresponding public commitment shares:

$$
(D_{ij}, E_{ij}) = (g^{d_{ij}}, g^{e_{ij}}).
$$

The private values are stored for later signing, while the public commitments are published.

## Protocol: Preprocess($\pi$)

Each participant $P_i$, for $i \in \{1, \dots, n\}$, performs the following steps before signing.

### 1. Create an empty commitment list

Participant $P_i$ starts with an empty list $L_i$.

### 2. For each counter $j = 1, \dots, \pi$

For each future signing slot $j$:

1. Sample single-use nonces

$$
(d_{ij}, e_{ij}) \xleftarrow{\$} \mathbb{Z}_q^* \times \mathbb{Z}_q^*
$$

2. Compute public commitment shares

$$
D_{ij} = g^{d_{ij}}, \qquad E_{ij} = g^{e_{ij}}
$$

3. Append the public pair $(D_{ij}, E_{ij})$ to $L_i$

4. Store the private/public tuples

$$
((d_{ij}, D_{ij}), (e_{ij}, E_{ij}))
$$

for later use in signing.

### 3. Publish the commitment list

After generating all $\pi$ pairs, participant $P_i$ publishes:

$$
(i, L_i)
$$

where

$$
L_i = \langle (D_{ij}, E_{ij}) \rangle_{j=1}^{\pi}.
$$

These are the public commitment shares that a signature aggregator may later fetch when forming a signing session.

## Why Two Nonces Per Participant?

FROST uses two nonce values per participant, not one. Later, during signing, these are combined with a **binding value** $\rho_i$ to form an effective nonce contribution:

$$
k_i = d_i + e_i \rho_i.
$$

That scalar contribution then becomes a group commitment contribution:

$$
R_i = D_i \cdot E_i^{\rho_i} = g^{d_i} \cdot (g^{e_i})^{\rho_i} = g^{d_i + e_i\rho_i} = g^{k_i}.
$$

Across the signing subset $S$, the scalar nonce contributions add to

$$
k = \sum_{i \in S} k_i,
$$

and the corresponding group commitment is

$$
R = g^k = \prod_{i \in S} R_i.
$$

So the relationship is not $\sum_i k_i = R$; rather, the nonce scalars add in
$\mathbb{Z}_q$, and their commitment points multiply in the group $G$.

That extra structure is what helps FROST bind each participant's response to:

- the specific message being signed
- the exact set of signing participants
- the exact commitment values used in that session

This is a major part of how FROST avoids known concurrency attacks without giving up efficiency.

## Why the Binding Matters

The paper highlights an attack direction discussed by Drijvers et al.: if an adversary can adaptively choose messages or commitments across many simultaneous sessions, they may try to manipulate the resulting challenge values.

FROST prevents this by binding a participant’s response to the specific signing context. That means partial responses from different messages or different commitment sets cannot be mixed together into a valid signature.

So preprocessing is not just a performance trick. It is part of a signing design that keeps concurrency safe.

## Operational Meaning

The published tuples $(i, L_i)$ are stored somewhere the signing workflow can access them. Depending on the implementation, that might be:

- a commitment server
- a coordinator database
- a shared bulletin-board style service
- or direct peer-to-peer exchange in a simpler deployment

The main requirement is that later, during signing, the system can retrieve a valid unused commitment pair for each participant chosen for that signing session.

## Important Safety Rule

Each nonce pair $(d_{ij}, e_{ij})$ is **single use**. Once used in signing, it must never be reused.

This is a critical security property. Reusing nonce material in Schnorr-style protocols can leak secret information, and in threshold settings that can expose a participant’s long-lived secret share.

So preprocessing must be stateful: participants need to know which commitment pairs are still unused and which have already been consumed.